# `DataValidator` - Validation des données agricoles

**Module :** `kadi.kidas.validator`  
**Classe :** `DataValidator`

---

## Introduction

`DataValidator` est la classe de validation de la qualité des données agricoles. Elle vérifie :

- **Schéma** : présence et types des colonnes obligatoires
- **Plages de valeurs** : rendements, prix, températures dans des bornes agronomiques
- **Coordonnées GPS** : dans la bounding box du Bénin (`lat: 2.5–12.5, lon: 0.8–3.8`)
- **Unicité** : pas de doublons sur des combinaisons clés (`marche + date + culture`)
- **Score de qualité global** : composites sur 3 dimensions (complétude, cohérence, précision)
- **Rapport de validation** : synthèse consultable après toutes les vérifications

## 1. Données de démonstration

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import numpy as np
import pandas as pd

from kadi.kidas.validator import DataValidator

# Dataset agricole réaliste (prix marchés Bénin)
df = pd.DataFrame({
    'culture':         ['maïs', 'niébé', 'igname', 'sorgho', 'manioc'],
    'marche':          ['Dantokpa', 'Parakou', 'Bohicon', 'Kandi', 'Cotonou'],
    'prix_xof':        [350, 500, 250, 300, 180],
    'rendement_kg':    [1500.0, 800.0, 2000.0, 1200.0, 3500.0],
    'lat':             [6.366, 9.337, 7.181, 11.133, 6.366],
    'lon':             [2.437, 2.629, 2.067, 2.740, 2.420],
    'date':            pd.to_datetime(['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19']),
})

# Dataset avec erreurs intentionnelles pour la démonstration
df_avec_erreurs = pd.DataFrame({
    'culture':         ['maïs', 'niébé', None, 'sorgho'],
    'prix_xof':        [350, -50, 99999, 300],       # -50 et 99999 : invalides
    'lat':             [6.366, 9.337, 50.0, 11.133],  # 50.0 : hors bbox Bénin
    'lon':             [2.437, 2.629, 2.067, 2.740],
})

print(f"Dataset valide : {len(df)} lignes")
df

Dataset valide : 5 lignes


,culture,marche,prix_xof,rendement_kg,lat,lon,date
0,maïs,Dantokpa,350,1500.0,6.366,2.437,2024-01-15
1,niébé,Parakou,500,800.0,9.337,2.629,2024-01-16
2,igname,Bohicon,250,2000.0,7.181,2.067,2024-01-17
3,sorgho,Kandi,300,1200.0,11.133,2.740,2024-01-18
4,manioc,Cotonou,180,3500.0,6.366,2.420,2024-01-19


## 2. Initialisation de `DataValidator`

In [2]:
validator = DataValidator(df)
print(repr(validator))

## 3. `validate_schema()` - Validation du schéma de données

In [3]:
# Définition du schéma attendu : nom_colonne → type
schema_attendu = {
    'culture':      'str',
    'marche':       'str',
    'prix_xof':     'float',
    'rendement_kg': 'float',
    'lat':          'float',
    'lon':          'float',
    'date':         'datetime',
}

# Validation du schéma
est_valide, erreurs = validator.validate_schema(schema_attendu)
print(f"Schéma valide : {est_valide}")
if erreurs:
    for e in erreurs:
        print(f"  - {e}")
else:
    print("  Aucune erreur de schéma !")

Schéma valide : False
  - Type incorrect pour 'prix_xof' : attendu 'float', reçu 'int64'.


In [4]:
# Test avec une colonne manquante
schema_invalide = {
    'culture': 'str',
    'colonne_manquante': 'float',  # N'existe pas
    'prix_xof': 'int',             # Mauvais type (c'est un float)
}

validator2 = DataValidator(df)
valide, erreurs = validator2.validate_schema(schema_invalide)
print(f"Schéma invalide : {valide}")
for e in erreurs:
    print(f"  - {e}")

Schéma invalide : False
  - Colonne manquante : 'colonne_manquante' (type attendu : 'float').


## 4. `validate_ranges()` - Validation des plages de valeurs agronomiques

In [5]:
# Définition des plages acceptables pour les données agricoles Bénin
plages = {
    'prix_xof':     (0, 10000),      # Prix : entre 0 et 10 000 XOF/kg
    'rendement_kg': (0, 50000),      # Rendement : entre 0 et 50 tonnes/ha
}

validator = DataValidator(df)
est_valide, df_hors_bornes = validator.validate_ranges(plages)
print(f"Toutes valeurs dans les plages : {est_valide}")
if len(df_hors_bornes) > 0:
    print(df_hors_bornes)

Toutes valeurs dans les plages : True


In [6]:
# Test avec des données erronées
validator_err = DataValidator(df_avec_erreurs)
est_valide_err, df_hors = validator_err.validate_ranges({'prix_xof': (0, 5000)})
print(f"Valeurs hors bornes : {est_valide_err}")
print(df_hors[['culture', 'prix_xof']])

2 valeur(s) hors intervalle [0.00, 5000.00] dans 'prix_xof'.


Valeurs hors bornes : False
  culture  prix_xof
1   niébé       -50
2     NaN     99999


## 5. `validate_coordinates()` - Validation des coordonnées GPS (bbox Bénin)

In [7]:
# Bbox Bénin par défaut : lat ∈ [2.5, 12.5], lon ∈ [0.8, 3.8]
validator = DataValidator(df)
est_valide, df_invalides = validator.validate_coordinates('lat', 'lon')
print(f"Coordonnées dans la bbox Bénin : {est_valide}")
if len(df_invalides) == 0:
    print("  Tous les points GPS sont dans le territoire béninois.")

Coordonnées dans la bbox Bénin : True
  Tous les points GPS sont dans le territoire béninois.


In [8]:
# Test avec des coordonnées hors Bénin (lat=50.0 = Europe)
validator_err = DataValidator(df_avec_erreurs)
est_valide_err, df_hors_bbox = validator_err.validate_coordinates('lat', 'lon')
print(f"Coordonnées hors bbox Bénin : {est_valide_err}")
print(f"Lignes concernées : {len(df_hors_bbox)}")
if len(df_hors_bbox) > 0:
    print(df_hors_bbox[['culture', 'lat', 'lon']])

1 coordonnée(s) hors bbox benin détectée(s). Vérifiez les inversions lat/lon éventuelles.


Coordonnées hors bbox Bénin : False
Lignes concernées : 1
  culture   lat    lon
2     NaN  50.0  2.067


In [9]:
# Validation avec une bbox personnalisée (zone nord Bénin uniquement)
validator = DataValidator(df)
est_valide_nord, _ = validator.validate_coordinates(
    'lat', 'lon',
    lat_bounds=(9.0, 12.5),
    lon_bounds=(1.5, 3.5),
)
print(f"Uniquement dans le nord Bénin : {est_valide_nord}")

TypeError: DataValidator.validate_coordinates() got an unexpected keyword argument 'lat_bounds'

## 6. `validate_uniqueness()` - Validation de l'unicité des combinaisons clés

In [10]:
# Vérification de l'unicité de la combinaison (marche + culture)
validator = DataValidator(df)
est_unique, df_doublons = validator.validate_uniqueness(['marche', 'culture'])
print(f"Combinaison (marche, culture) unique : {est_unique}")
if len(df_doublons) > 0:
    print(df_doublons[['marche', 'culture']])

Combinaison (marche, culture) unique : True


In [11]:
# Démonstration avec un DataFrame contenant des doublons
df_dup = pd.concat([df, df.iloc[[0]]], ignore_index=True)
validator_dup = DataValidator(df_dup)
est_unique_dup, df_dup_detectes = validator_dup.validate_uniqueness(['marche', 'culture'])
print(f"Combinaison unique (avec doublons) : {est_unique_dup}")
print(f"Lignes dupliquées : {len(df_dup_detectes)}")

Combinaison unique (avec doublons) : False
Lignes dupliquées : 2


## 7. `compute_quality_score()` - Score de qualité global

Le score est calculé sur **3 dimensions** :

| Dimension | Calcul |
|---|---|
| Complétude | `1 - (NaN / total)` |
| Cohérence | Ratio colonnes types corrects |
| Précision | Ratio valeurs dans les plages |

Le score global est la **moyenne pondérée** des 3 dimensions.

In [12]:
validator = DataValidator(df)
score = validator.compute_quality_score()

print("=== Score de Qualité ===")
print(f"Score global     : {score['overall']:.2%}")
print(f"Complétude       : {score['completeness']:.2%}")
print(f"Cohérence        : {score['consistency']:.2%}")
print(f"Précision        : {score['accuracy']:.2%}")
print(f"\nDétail par colonne :")
for col, detail in score['columns'].items():
    print(f"  {col:15s} : complétude={detail['completeness']:.2%}")

=== Score de Qualité ===
Score global     : 100.00%
Complétude       : 100.00%
Cohérence        : 100.00%
Précision        : 100.00%

Détail par colonne :


TypeError: 'float' object is not subscriptable

In [13]:
# Comparaison avec les données erronées
validator_err = DataValidator(df_avec_erreurs)
score_err = validator_err.compute_quality_score()
print(f"Score global (données erronées) : {score_err['overall']:.2%}")
print(f"Complétude                      : {score_err['completeness']:.2%}")

Score global (données erronées) : 97.50%
Complétude                      : 93.75%


## 8. `get_validation_report()` - Rapport complet de validation

In [14]:
# Exécution de plusieurs validations avant de générer le rapport
validator = DataValidator(df)
validator.validate_schema({'culture': 'str', 'prix_xof': 'float'})
validator.validate_ranges({'prix_xof': (0, 10000)})
validator.validate_coordinates('lat', 'lon')
validator.compute_quality_score()

rapport = validator.get_validation_report()

print("=== Rapport de Validation ===")
print(f"Validations effectuées : {len(rapport['validations'])}")
for v in rapport['validations']:
    print(f"  - {v['type']:20s} : {'OK' if v['valid'] else 'ECHEC'}")

if 'quality_score' in rapport:
    print(f"\nScore global : {rapport['quality_score']['overall']:.2%}")

=== Rapport de Validation ===
Validations effectuées : 3


KeyError: 'valid'

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_schema(schema)` | Vérifie présence et types des colonnes |
| `validate_ranges(ranges)` | Détecte les valeurs hors bornes agronomiques |
| `validate_coordinates(lat, lon, ...)` | Valide les GPS contre la bbox Bénin |
| `validate_uniqueness(columns)` | Détecte les doublons sur combinaisons clés |
| `compute_quality_score()` | Score composite 0–1 (complétude, cohérence, précision) |
| `get_validation_report()` | Synthèse de toutes les validations effectuées |